# 🍃 LeafScan: MobileNetV3-Small Dual-Head Leaf Disease Detector
### Google Colab Training & Evaluation Notebook

This notebook trains a lightweight, dual-head **MobileNetV3-Small** deep learning model for vegetable leaf disease classification.
- **Vegetable Head (4 classes)**: Broccoli, Cabbage, Cauliflower, Turnip
- **Disease Head (17 classes)**: 17 disease/healthy conditions from the Mendeley dataset
- **Novelty & Features**: Stratified 70/15/15 split, class weighting for imbalance, Grad-CAM interpretability, <60% confidence fallback, TF.js export.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip install -q tensorflow scikit-learn matplotlib seaborn pillow tensorflowjs

In [ ]:
import os
import sys
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow Version:", tf.__version__)

## 2. Dataset Setup & Parsing
Upload your dataset zip file or mount Google Drive. The dataset should contain subfolders for each class (e.g. `Broccoli Alternaria leaf spot`, `Cauliflower Yellow virus`, etc.).

In [ ]:
# Set dataset path
DATA_DIR = "dataset"  # Update this path if dataset is unzipped elsewhere (e.g., /content/dataset)
VEGETABLE_CLASSES = ["Broccoli", "Cabbage", "Cauliflower", "Turnip"]

def parse_vegetable_and_disease(folder_name):
    cleaned = folder_name.strip()
    for veg in VEGETABLE_CLASSES:
        if cleaned.lower().startswith(veg.lower()):
            disease = cleaned[len(veg):].strip()
            if not disease:
                disease = "Healthy leaf"
            return veg, disease
    parts = cleaned.split(" ", 1)
    return parts[0], parts[1] if len(parts) > 1 else "Unknown"

# Scan directory
subdirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
class_names = subdirs
veg_names = sorted(list(set([parse_vegetable_and_disease(d)[0] for d in class_names])))

class_to_idx = {name: i for i, name in enumerate(class_names)}
veg_to_idx = {name: i for i, name in enumerate(veg_names)}

image_paths, disease_labels, vegetable_labels = [], [], []
valid_exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
for c_name in class_names:
    c_dir = os.path.join(DATA_DIR, c_name)
    veg_name, _ = parse_vegetable_and_disease(c_name)
    files = []
    for ext in valid_exts:
        files.extend(glob.glob(os.path.join(c_dir, ext)))
    for f in files:
        image_paths.append(f)
        disease_labels.append(class_to_idx[c_name])
        vegetable_labels.append(veg_to_idx[veg_name])

print(f"Found {len(image_paths)} images across {len(class_names)} disease classes and {len(veg_names)} vegetable categories.")

## 3. Stratified 70/15/15 Split & Class Weighting

In [ ]:
# Stratified 70% Train, 15% Validation, 15% Test
paths_train, paths_temp, y_dis_train, y_dis_temp, y_veg_train, y_veg_temp = train_test_split(
    image_paths, disease_labels, vegetable_labels,
    test_size=0.30,
    stratify=disease_labels,
    random_state=SEED
)

paths_val, paths_test, y_dis_val, y_dis_test, y_veg_val, y_veg_test = train_test_split(
    paths_temp, y_dis_temp, y_veg_temp,
    test_size=0.50,
    stratify=y_dis_temp,
    random_state=SEED
)

print(f"Train samples: {len(paths_train)}, Val samples: {len(paths_val)}, Test samples: {len(paths_test)}")

# Balanced class weights for handling dataset imbalance
cw_arr = compute_class_weight(class_weight="balanced", classes=np.unique(y_dis_train), y=y_dis_train)
disease_class_weights = {i: float(w) for i, w in enumerate(cw_arr)}
print("Sample class weights:", {k: round(v, 2) for k, v in list(disease_class_weights.items())[:5]})

## 4. `tf.data` Pipeline with Augmentations

In [ ]:
num_classes = len(class_names)
num_vegs = len(veg_names)
BATCH_SIZE = 32

def load_and_preprocess(path, dis_lbl, veg_lbl):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [224, 224])
    return img, {
        "vegetable_output": tf.one_hot(veg_lbl, depth=num_vegs),
        "disease_output": tf.one_hot(dis_lbl, depth=num_classes)
    }

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

def create_dataset(paths, y_d, y_v, is_train=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, y_d, y_v))
    if is_train:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if is_train:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(paths_train, y_dis_train, y_veg_train, is_train=True)
val_ds = create_dataset(paths_val, y_dis_val, y_veg_val, is_train=False)
test_ds = create_dataset(paths_test, y_dis_test, y_veg_test, is_train=False)

## 5. MobileNetV3-Small Dual-Head Model Architecture

In [ ]:
inputs = tf.keras.layers.Input(shape=(224, 224, 3), name="input_layer")
preprocessed = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)

backbone = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg",
    include_preprocessing=False
)

features = backbone(preprocessed)
features = tf.keras.layers.BatchNormalization(name="shared_bn")(features)
features = tf.keras.layers.Dropout(0.3, name="shared_dropout")(features)

# Head 1: Vegetable Head
veg_dense = tf.keras.layers.Dense(64, activation="relu", name="veg_fc")(features)
veg_out = tf.keras.layers.Dense(num_vegs, activation="softmax", name="vegetable_output")(veg_dense)

# Head 2: Disease Head
dis_dense = tf.keras.layers.Dense(128, activation="relu", name="disease_fc")(features)
dis_out = tf.keras.layers.Dense(num_classes, activation="softmax", name="disease_output")(dis_dense)

model = tf.keras.Model(inputs=inputs, outputs={"vegetable_output": veg_out, "disease_output": dis_out}, name="LeafScan_Model")
model.summary()

## 6. Two-Stage Training (Stage 1: Frozen Backbone, Stage 2: Fine-Tuning)

In [ ]:
losses = {"vegetable_output": "categorical_crossentropy", "disease_output": "categorical_crossentropy"}
loss_weights = {"vegetable_output": 0.3, "disease_output": 1.0}
metrics = {"vegetable_output": ["accuracy"], "disease_output": ["accuracy"]}

# --- STAGE 1 ---
backbone.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=losses, loss_weights=loss_weights, metrics=metrics)

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_disease_output_accuracy", patience=5, restore_best_weights=True)],
    class_weight={"disease_output": disease_class_weights}
)

# --- STAGE 2 ---
backbone.trainable = True
fine_tune_at = int(len(backbone.layers) * 0.7)
for layer in backbone.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=losses, loss_weights=loss_weights, metrics=metrics)

os.makedirs("results", exist_ok=True)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "results/best_leafscan_model.keras",
    monitor="val_disease_output_accuracy",
    save_best_only=True,
    mode="max"
)

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_disease_output_accuracy", patience=6, restore_best_weights=True),
        checkpoint_cb
    ],
    class_weight={"disease_output": disease_class_weights}
)

## 7. Grad-CAM Interpretability Function

In [ ]:
import matplotlib.cm as cm

def compute_gradcam(model, img_array, pred_index=None):
    # Locate last convolutional layer of backbone
    conv_layer = backbone.get_layer("Conv_1") if "Conv_1" in [l.name for l in backbone.layers] else backbone.layers[-4]
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[conv_layer.output, model.get_layer("disease_output").output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    heatmap = heatmap.numpy()
    
    heatmap_img = Image.fromarray(np.uint8(255 * heatmap)).resize((224, 224), resample=Image.BICUBIC)
    return np.array(heatmap_img) / 255.0

def overlay_gradcam(img_np, heatmap, alpha=0.45):
    color_map = cm.get_cmap("jet")
    color_heatmap = np.uint8(255 * color_map(heatmap)[:, :, :3])
    return np.uint8(img_np * (1 - alpha) + color_heatmap * alpha)

## 8. Test Set Evaluation & Metrics Generation

In [ ]:
y_true_veg, y_true_dis, y_pred_veg, y_pred_dis, y_probs = [], [], [], [], []

for imgs, targets in test_ds:
    preds = model.predict(imgs, verbose=0)
    veg_t, dis_t = targets["vegetable_output"].numpy(), targets["disease_output"].numpy()
    veg_p, dis_p = preds["vegetable_output"], preds["disease_output"]
    
    y_true_veg.extend(np.argmax(veg_t, axis=1))
    y_true_dis.extend(np.argmax(dis_t, axis=1))
    y_pred_veg.extend(np.argmax(veg_p, axis=1))
    y_pred_dis.extend(np.argmax(dis_p, axis=1))
    y_probs.extend(np.max(dis_p, axis=1))

acc_veg = accuracy_score(y_true_veg, y_pred_veg)
acc_dis = accuracy_score(y_true_dis, y_pred_dis)
low_conf = sum(1 for p in y_probs if p < 0.60)

print(f"\n=== TEST SET RESULTS ===")
print(f"Vegetable Accuracy: {acc_veg * 100:.2f}%")
print(f"Disease Accuracy:   {acc_dis * 100:.2f}%")
print(f"Low Confidence (< 60%): {low_conf} / {len(y_probs)} ({low_conf / len(y_probs) * 100:.2f}%)")

# Confusion Matrix Plot
plt.figure(figsize=(14, 12))
sns.heatmap(confusion_matrix(y_true_dis, y_pred_dis), annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Disease Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("results/confusion_matrix_disease.png", dpi=300)
plt.show()